In [10]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & artiq_run " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, t_raman_pulse):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                # self.p.frequency_raman_transition =

                self.p.t_continuous_rabi = 300.e-6

                # self.xvar('t_raman_pulse',np.linspace(0.,8.7e-6,7))
                self.p.t_raman_pulse = {t_raman_pulse}
                
                self.xvar('amp_imaging',np.linspace(2.45,3.5, 10))
                # self.p.amp_imaging

                self.p.hf_imaging_detuning = -568.e6 # 182.

                self.p.hf_imaging_detuning_mid = -514.e6 # -1 --> 0
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.186 * np.pi

                self.p.t_tweezer_hold = 20.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 10

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning_mid)
                # self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)

                self.raman.pulse(t=self.p.t_raman_pulse)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                # delay(3.e-6)
                # self.raman.pulse(t=self.p.t_continuous_rabi)
                delay(self.p.t_continuous_rabi)
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [11]:
eBuilder = ExptBuilder()

In [12]:
raman_pulse = [0.,9.3517e-06 / 2, 9.3517e-06]
for f in range(3):
    print(raman_pulse[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(t_raman_pulse=raman_pulse[f]))
    eBuilder.run_expt()

0.0
0 Run id: 63564
 100 values of amp_imaging. 100 total shots. 300 total images expected.
B:\_K\PotassiumData\2026-04-12\0063564_2026-04-12_11-36-54_hf_monitored_rabi.hdf5
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820

 Run ID: 63564

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 

In [13]:

relay = ethernet_relay

In [14]:
relay.source_off()

AttributeError: module 'waxx.control.ethernet_relay' has no attribute 'source_off'